# 3.Регрессионные модели
**Выполнено обучение моделей XGBoost, Random Forest,CatBoost,LightGBM для предсказания логарифма IC50. Для улучшения значения R² проведена настройка гиперпараметров с использованием метода случайного поиска (RandomizedSearchCV)**

In [ ]:
# подбор гиперпараметров XGBoost для IC50

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_ic50.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_ic50.csv').values.ravel()

xgb = XGBRegressor(random_state=42, verbosity=0, early_stopping_rounds=10, eval_metric='rmse')

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

best_model = random_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("XGBoost Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {random_search.best_params_}")

joblib.dump(best_model, 'xgboost_ic50.pkl')

XGBoost Results:
R2: 0.4383
MAE_log: 1.1434
RMSE_log: 1.4295
MAE_original: 184.01
Best params: {'subsample': 0.7, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.7}


['xgboost_ic50.pkl']

In [ ]:
# подбор гиперпараметров Random Forest для IC50

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_ic50.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_ic50.csv').values.ravel()

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_params,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_search.fit(X_train, y_train)

best_model = rf_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("Random Forest Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {rf_search.best_params_}")

joblib.dump(best_model, 'randomforest_ic50.pkl')

Random Forest Results:
R2: 0.4261
MAE_log: 1.1470
RMSE_log: 1.4450
MAE_original: 184.70
Best params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}


['randomforest_ic50.pkl']

In [ ]:
!pip install catboost lightgbm xgboost

In [ ]:
# подбор гиперпараметров CatBoost для IC50

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_ic50.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_ic50.csv').values.ravel()

cat = CatBoostRegressor(random_seed=42, verbose=False, early_stopping_rounds=10)

param_dist_cat = {
    'iterations': [100, 200, 300, 500],
    'depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'l2_leaf_reg': [1, 3, 5, 7, 10]
}

cat_search = RandomizedSearchCV(
    estimator=cat,
    param_distributions=param_dist_cat,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

cat_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

best_model = cat_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("CatBoost Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {cat_search.best_params_}")

joblib.dump(best_model, 'catboost_ic50.pkl')

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


CatBoost Results:
R2: 0.4468
MAE_log: 1.1312
RMSE_log: 1.4187
MAE_original: 181.08
Best params: {'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 500, 'depth': 7}


['catboost_ic50.pkl']

In [ ]:
# подбор гиперпараметров LightGBM для IC50

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

X_train = pd.read_csv('/content/X_train.csv')
X_test = pd.read_csv('/content/X_test.csv')
y_train = pd.read_csv('/content/y_train_ic50.csv').values.ravel()
y_test = pd.read_csv('/content/y_test_ic50.csv').values.ravel()

lgbm = lgb.LGBMRegressor(random_state=42, verbose=-1)

param_dist_lgb = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [3, 5, 7, 9, 12, -1],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'num_leaves': [31, 63, 127, 255, 511],
    'min_child_samples': [5, 10, 20, 30],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0, 0.1, 0.5, 1.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist_lgb,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

lgb_search.fit(X_train, y_train)

best_model = lgb_search.best_estimator_
y_pred_log = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred_log)
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

y_pred_original = np.expm1(y_pred_log)
y_test_original = np.expm1(y_test)
mae_original = mean_absolute_error(y_test_original, y_pred_original)

print("LightGBM Results:")
print(f"R2: {r2:.4f}")
print(f"MAE_log: {mae_log:.4f}")
print(f"RMSE_log: {rmse_log:.4f}")
print(f"MAE_original: {mae_original:.2f}")
print(f"Best params: {lgb_search.best_params_}")

joblib.dump(best_model, 'lightgbm_ic50.pkl')

LightGBM Results:
R2: 0.4288
MAE_log: 1.1330
RMSE_log: 1.4416
MAE_original: 186.90
Best params: {'subsample': 0.9, 'reg_lambda': 0.1, 'reg_alpha': 0.1, 'num_leaves': 255, 'n_estimators': 100, 'min_child_samples': 30, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.6}


['lightgbm_ic50.pkl']

In [ ]:
# сравнение моделей для IC50
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

# загрузка тестовых данных
X_test = pd.read_csv('/content/X_test.csv')
y_test = pd.read_csv('/content/y_test_ic50.csv').values.ravel()

# загрузка моделей
xgb_model = joblib.load('xgboost_ic50.pkl')
rf_model = joblib.load('randomforest_ic50.pkl')
cat_model = joblib.load('catboost_ic50.pkl')
lgb_model = joblib.load('lightgbm_ic50.pkl')

print("="*60)
print("Сравнение моделей на тестовой выборке")
print("="*60)

# функция для расчёта метрик
def evaluate_model(model, name):
    y_pred_log = model.predict(X_test)

    r2 = r2_score(y_test, y_pred_log)
    mae_log = mean_absolute_error(y_test, y_pred_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))

    y_pred_orig = np.expm1(y_pred_log)
    y_test_orig = np.expm1(y_test)
    mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)

    print(f"\n{name}")
    print(f"  R2: {r2:.4f}")
    print(f"  MAE_log: {mae_log:.4f}")
    print(f"  RMSE_log: {rmse_log:.4f}")
    print(f"  MAE_original (нМ): {mae_orig:.2f}")

# оцениваем каждую модель
evaluate_model(xgb_model, "XGBoost")
evaluate_model(rf_model, "Random Forest")
evaluate_model(cat_model, "CatBoost")
evaluate_model(lgb_model, "LightGBM")

print("\n" + "="*60)
print("Лучшая модель по R2 и MAE_original:")
# поиск лучшей вручную
models_list = [
    ("XGBoost", xgb_model),
    ("Random Forest", rf_model),
    ("CatBoost", cat_model),
    ("LightGBM", lgb_model)
]

best_r2 = -np.inf
best_mae = np.inf
best_name_r2 = ""
best_name_mae = ""

for name, model in models_list:
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    y_pred_orig = np.expm1(y_pred)
    y_test_orig = np.expm1(y_test)
    mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)

    if r2 > best_r2:
        best_r2 = r2
        best_name_r2 = name
    if mae_orig < best_mae:
        best_mae = mae_orig
        best_name_mae = name

print(f"  Лучшая по R2: {best_name_r2} ({best_r2:.4f})")
print(f"  Лучшая по MAE(нМ): {best_name_mae} ({best_mae:.2f})")

Сравнение моделей на тестовой выборке

XGBoost
  R2: 0.4383
  MAE_log: 1.1434
  RMSE_log: 1.4295
  MAE_original (нМ): 184.01

Random Forest
  R2: 0.4261
  MAE_log: 1.1470
  RMSE_log: 1.4450
  MAE_original (нМ): 184.70

CatBoost
  R2: 0.4468
  MAE_log: 1.1312
  RMSE_log: 1.4187
  MAE_original (нМ): 181.08

LightGBM
  R2: 0.4288
  MAE_log: 1.1330
  RMSE_log: 1.4416
  MAE_original (нМ): 186.90

Лучшая модель по R2 и MAE_original:
  Лучшая по R2: CatBoost (0.4468)
  Лучшая по MAE(нМ): CatBoost (181.08)


Вывод:лучшая модель: CatBoost (R 2
 =0.4468,
MAE
ориг
​
 =181.08 нМ).

Результат: Модели показывают умеренную предсказательную способность. Значение
R
2
≈
0.44
 означает, что модель объясняет чуть меньше половины дисперсии целевой переменной. Это неплохой старт для химиоинформатической задачи, где данные часто имеют сложную, нелинейную природу. Ошибка в ~180 нМ на оригинальной шкале говорит о том, что модель может различать соединения с низкой и высокой активностью, но для точного количественного предсказания требуется дальнейшее улучшение.